# Tutorial 13-3: The Sinusoidal Fingerprint
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. Why Not Just 1, 2, 3?
Self-attention is an unordered function—it treats a sentence like a 'bag of words' unless we explicitly add order information. You might wonder: why not just add the index (0, 1, 2...) to the word embedding?

**The Problems:**
1. **Magnitude:** In long sequences, the index values could become massive, overshadowing the actual word embeddings.
2. **Generalization:** If a model only saw sequences of length 50 during training, it wouldn't know what to do with index 51.

**The Solution:** Sinusoidal Positional Encoding. By using sine and cosine waves of different frequencies, we create a unique, bounded vector for every position.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def get_positional_encoding(max_seq_len, d_model):
    """Implementation of the formula from the slides to generate the positional encoding matrix."""
    pe = np.zeros((max_seq_len, d_model))
    for pos in range(max_seq_len):
        for i in range(0, d_model, 2):
            div_term = np.exp(i * -(np.log(10000.0) / d_model))
            pe[pos, i] = np.sin(pos * div_term)
            pe[pos, i + 1] = np.cos(pos * div_term)
    return pe

# Parameters for visualization
L = 100   # Sequence length
D = 128   # Embedding dimension

pe_matrix = get_positional_encoding(L, D)

print(f"Generated Positional Encoding Matrix: {pe_matrix.shape}")

## 2. Visualizing the Encoding Matrix
The heatmap below shows the 'fingerprint' BERT or GPT would add to its inputs. Each row is the vector added to a word at that position. Notice the alternating patterns of high and low frequencies.

In [ ]:
plt.figure(figsize=(12, 8))
plt.pcolormesh(pe_matrix, cmap='RdBu')
plt.xlabel('Embedding Dimension (d_model)')
plt.ylabel('Position in Sequence')
plt.colorbar(label='Encoding Value')
plt.title("Visualizing the Positional Encoding Matrix")
plt.show()

## 3. High vs. Low Frequency Dimensions
Let's look at specific dimensions. Some dimensions change rapidly (high frequency), helping the model distinguish between words that are right next to each other. Others change slowly, helping the model understand the broader 'arc' of the sentence.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(pe_matrix[:, 0], label="Dim 0 (High Frequency)")
plt.plot(pe_matrix[:, 20], label="Dim 20")
plt.plot(pe_matrix[:, 60], label="Dim 60 (Low Frequency)")
plt.legend()
plt.title("Sine Waves Across Different Dimensions")
plt.xlabel("Position index")
plt.ylabel("Value")
plt.grid(True)
plt.show()

## 4. Key Takeaways
* **Uniqueness:** Every position has a unique vector.
* **Boundedness:** Values stay between -1 and 1, so they don't dominate the actual word information.
* **Relative Position:** Because of trigonometric identities, the model can learn that $PE_{pos+k}$ is a linear function of $PE_{pos}$. This allows the attention mechanism to attend to 'the word 3 positions to my left' easily.